# Bubble method comparison (GSADF vs LPPLS vs CUSUM)

This notebook runs the same logic as **`bubble_method_comparison.py`** by calling **`run_comparison()`**.
The `.py` file remains the single source of truth for helpers and the CLI (`python bubble_method_comparison.py`).

**Inputs** (under repo `R/` or `R/data_R/`): prefer `df_master_gsadf.csv`, `df_master_lppls.csv`, `df_master_cusum.csv`.

**Outputs:** `outputs/bubble_methods_*.csv` and PNG figures (same as the script).

## 1. Load the comparison module

Works whether the Jupyter **working directory** is the **metals** repo root or **`notebooks/analysis`**.

In [ ]:
import importlib.util
from pathlib import Path

_candidates = []
for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents[:6]]:
    _candidates.extend(
        [
            base / "notebooks" / "analysis" / "bubble_method_comparison.py",
            base / "bubble_method_comparison.py",
        ]
    )

_mod_path = next((p for p in _candidates if p.is_file()), None)
if _mod_path is None:
    raise FileNotFoundError(
        "bubble_method_comparison.py not found. Open the notebook from the metals repo or cd to notebooks/analysis."
    )

_spec = importlib.util.spec_from_file_location("bubble_method_comparison", _mod_path)
bmc = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(bmc)
print("Loaded:", _mod_path)

## 2. Run comparison

- Set **`RECOMPUTE_CUSUM = True`** to ignore `df_master_cusum.csv` and recompute CUSUM from the GSADF master (slower).
- Set **`OUTPUT_DIR`** to a string path to override the default `outputs/` folder.

In [ ]:
RECOMPUTE_CUSUM = False
OUTPUT_DIR = ""  # e.g. r"C:\temp\bubble_out" or leave "" for repo outputs/

result = bmc.run_comparison(
    recompute_cusum=RECOMPUTE_CUSUM,
    output_dir=OUTPUT_DIR or None,
)

## 3. Inspect results in memory

In [ ]:
from IPython.display import display

print("Output directory:", result["out_dir"])
print("Files:", result["paths"])
display(result["prevalence"])
display(result["pairwise"].head(20))
if result["kappa"] is not None:
    display(result["kappa"])
else:
    print("(Install scikit-learn for Cohen kappa table)")